# HNSW From Scratch

Companion notebook for the [Vector Search Algorithms](https://codemia.io/courses/applied_ai_systems/vector_search_algorithms) lesson on Codemia.

**What you will build**: a minimal implementation of Hierarchical Navigable Small World graphs (HNSW) — no FAISS, no hnswlib — that achieves >90% recall against exact kNN on a toy dataset.

**Why from scratch**: production libraries are fast and opaque. Writing HNSW yourself forces you to understand the layered graph, the greedy search, and the tradeoffs of `M`, `efConstruction`, and `efSearch`.

**References**:
- Malkov & Yashunin, [Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs](https://arxiv.org/abs/1603.09320) (2016)
- Pinecone's [HNSW explainer](https://www.pinecone.io/learn/series/faiss/hnsw/)

**Time**: ~45 minutes  **GPU required**: no

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import heapq
import random
from collections import defaultdict

np.random.seed(0)
random.seed(0)

## Concept recap

HNSW stores vectors in a stack of graphs. The top layer has a handful of points connected in a very sparse graph — think "highways". As you descend, each layer adds more points and more edges until the bottom layer contains every vector. Search starts at the top entry point, greedily walks toward the query, then uses that endpoint as the entry for the layer below. The effect is skip-list-like: you cover distance quickly at the top, then refine at the bottom.

Two knobs matter:
- `M`: max neighbors per node per layer (graph degree). Bigger `M` = more recall, more memory.
- `efSearch`: size of the dynamic candidate list at query time. Bigger `efSearch` = higher recall, slower queries.

We will implement a bare-bones version and measure recall against exact kNN.

## Implementation

We start with a distance function and a priority-queue-based search over a single layer. Then we stack layers.

In [ ]:
def l2(a, b):
    # Squared L2 is fine for ranking — cheaper than sqrt and preserves order.
    # We only care about *relative* distances during search.
    diff = a - b
    return float(np.dot(diff, diff))

In [ ]:
class HNSW:
    def __init__(self, dim, M=8, ef_construction=32, m_L=None):
        self.dim = dim
        self.M = M                          # max neighbors per node per layer
        self.M_max0 = M * 2                  # layer 0 allows more neighbors — standard HNSW trick
        self.ef_construction = ef_construction
        # Level multiplier controls layer distribution. Paper recommends 1/ln(M).
        self.m_L = m_L if m_L is not None else 1.0 / math.log(M)
        self.data = []                       # list of vectors, indexed by internal id
        self.graphs = []                     # graphs[layer] is dict: node_id -> list of neighbor_ids
        self.entry_point = None              # top-layer entry node
        self.top_layer = -1

    def _random_level(self):
        # Exponentially decaying level assignment: most nodes get level 0,
        # fewer get level 1, etc. This is what gives HNSW its skip-list shape.
        return int(-math.log(random.random()) * self.m_L)

In [ ]:
def _search_layer(self, q, entry_points, ef, layer):
    # Greedy best-first search on one layer.
    # `candidates` is a min-heap of (distance, node_id): nodes we might still expand.
    # `results` is a max-heap of (-distance, node_id): the best ef we have found.
    # Using a max-heap for results lets us prune anything worse than the current worst.
    visited = set(entry_points)
    candidates = []
    results = []
    for ep in entry_points:
        d = l2(q, self.data[ep])
        heapq.heappush(candidates, (d, ep))
        heapq.heappush(results, (-d, ep))

    while candidates:
        d_c, c = heapq.heappop(candidates)
        # Early termination: if the closest candidate is farther than the farthest result,
        # no unvisited neighbor can improve results.
        worst_in_results = -results[0][0]
        if d_c > worst_in_results:
            break
        for nb in self.graphs[layer].get(c, []):
            if nb in visited:
                continue
            visited.add(nb)
            d_nb = l2(q, self.data[nb])
            worst = -results[0][0] if results else float('inf')
            if d_nb < worst or len(results) < ef:
                heapq.heappush(candidates, (d_nb, nb))
                heapq.heappush(results, (-d_nb, nb))
                if len(results) > ef:
                    heapq.heappop(results)  # drop farthest
    # Return as sorted list of (distance, node_id) ascending.
    return sorted([(-d, n) for d, n in results])

HNSW._search_layer = _search_layer

In [ ]:
def _select_neighbors(self, candidates, M):
    # Simplest heuristic: take the M closest from the candidate set.
    # The paper's better heuristic (selectNeighborsHeuristic) prunes edges that
    # go through existing neighbors to improve graph connectivity, but the simple
    # version is enough to see the core mechanics.
    return [n for _, n in sorted(candidates)[:M]]

HNSW._select_neighbors = _select_neighbors

In [ ]:
def insert(self, x):
    # 1. Assign a random level to this new node.
    level = self._random_level()
    node_id = len(self.data)
    self.data.append(x)

    # 2. If the graph is empty, this node becomes the entry point.
    if self.entry_point is None:
        for l in range(level + 1):
            self.graphs.append({node_id: []})
        self.entry_point = node_id
        self.top_layer = level
        return node_id

    # 3. Make sure we have enough layers. New layers start empty and get this
    #    node as the first entry.
    while len(self.graphs) <= level:
        self.graphs.append({})

    # 4. Top-down greedy search: descend from the top layer to level+1
    #    using ef=1 (just find the single nearest point as the entry for the next layer).
    ep = [self.entry_point]
    for l in range(self.top_layer, level, -1):
        nearest = self._search_layer(x, ep, ef=1, layer=l)
        ep = [nearest[0][1]]

    # 5. From min(level, top_layer) down to 0, search with ef_construction and connect.
    for l in range(min(level, self.top_layer), -1, -1):
        found = self._search_layer(x, ep, ef=self.ef_construction, layer=l)
        M = self.M_max0 if l == 0 else self.M
        neighbors = self._select_neighbors(found, M)
        # Add bidirectional edges.
        self.graphs[l][node_id] = list(neighbors)
        for nb in neighbors:
            nb_edges = self.graphs[l].setdefault(nb, [])
            nb_edges.append(node_id)
            # If neighbor's edge list is too long, prune to M closest.
            if len(nb_edges) > M:
                trimmed = self._select_neighbors(
                    [(l2(self.data[nb], self.data[e]), e) for e in nb_edges], M
                )
                self.graphs[l][nb] = trimmed
        ep = [n for _, n in found]

    # 6. Update global entry point if we rose above the current top layer.
    if level > self.top_layer:
        self.top_layer = level
        self.entry_point = node_id
    return node_id

HNSW.insert = insert

In [ ]:
def search(self, q, k=5, ef_search=32):
    # Top-down descent with ef=1, then a final ef_search pass at layer 0.
    ep = [self.entry_point]
    for l in range(self.top_layer, 0, -1):
        nearest = self._search_layer(q, ep, ef=1, layer=l)
        ep = [nearest[0][1]]
    found = self._search_layer(q, ep, ef=max(ef_search, k), layer=0)
    return found[:k]

HNSW.search = search

## Experiment

Build an index on 2000 random 64-dim vectors, then compare HNSW results against exact kNN.

In [ ]:
N, D = 2000, 64
X = np.random.randn(N, D).astype(np.float32)
queries = np.random.randn(50, D).astype(np.float32)

# Exact kNN baseline — O(N) per query.
def exact_knn(q, k=10):
    dists = ((X - q) ** 2).sum(axis=1)
    idx = np.argpartition(dists, k)[:k]
    return set(idx[np.argsort(dists[idx])].tolist())

# Build HNSW.
index = HNSW(dim=D, M=8, ef_construction=64)
for i in range(N):
    index.insert(X[i])
print(f'Built index with {len(index.data)} points, top layer = {index.top_layer}')

In [ ]:
# Recall@10 as a function of efSearch.
def recall_at_k(ef_search, k=10):
    hits = 0
    for q in queries:
        truth = exact_knn(q, k=k)
        got = set(n for _, n in index.search(q, k=k, ef_search=ef_search))
        hits += len(truth & got) / k
    return hits / len(queries)

for ef in [1, 4, 8, 16, 32, 64, 128]:
    r = recall_at_k(ef)
    print(f'efSearch={ef:4d}  recall@10 = {r:.3f}')

## Variations

Plot the efSearch/recall tradeoff, then try changing `M` to see how graph degree shifts the curve.

In [ ]:
efs = [1, 2, 4, 8, 16, 32, 64, 128]
recalls = [recall_at_k(ef) for ef in efs]
plt.figure(figsize=(6, 4))
plt.plot(efs, recalls, 'o-')
plt.xscale('log')
plt.xlabel('efSearch')
plt.ylabel('recall@10')
plt.title('HNSW recall vs efSearch')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rebuild with larger M and compare.
idx_big = HNSW(dim=D, M=16, ef_construction=64)
for i in range(N):
    idx_big.insert(X[i])

def recall_other(index, ef_search, k=10):
    hits = 0
    for q in queries:
        truth = exact_knn(q, k=k)
        got = set(n for _, n in index.search(q, k=k, ef_search=ef_search))
        hits += len(truth & got) / k
    return hits / len(queries)

for ef in [4, 16, 64]:
    print(f'ef={ef:3d}  M=8: {recall_at_k(ef):.3f}   M=16: {recall_other(idx_big, ef):.3f}')

Observations to notice:

1. Recall climbs rapidly with `efSearch` and saturates — there are diminishing returns above ~64 for this dataset size.
2. Bigger `M` means more edges at each node, so the bottom layer graph is better connected. You trade memory (and build time) for recall at low `efSearch`.
3. Even at `efSearch=1` we still return *something* — the greedy descent will always terminate at *some* node, just not necessarily the closest one.

## What to try next

- Implement the paper's `selectNeighborsHeuristic` (prunes edges that are redundant given closer neighbors) and measure the recall delta.
- Swap squared-L2 for cosine similarity. Because cosine is rank-equivalent to L2 on normalized vectors, you can simply L2-normalize all inputs first.
- Try a realistic embedding dataset: encode 10k Wikipedia paragraphs with `sentence-transformers/all-MiniLM-L6-v2` and index those 384-dim vectors.
- Measure query latency and compare to `faiss.IndexHNSWFlat` — see how much a real library wins (usually 10-100x from SIMD + better memory layout).
- Add deletion. Real HNSW libraries use tombstones — nodes marked deleted but left in the graph so neighbors' edges remain valid.